__Build a RAG System on the Bhagavad Gita or Bible__

In [1]:
!pip install langchain langchain-community pypdf langchain-text-splitters --quiet


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('Gita.pdf')
gita_book = loader.load()
print(f'Loaded {len(gita_book)} pages')
for page in gita_book[:2]:
    print(f'Page {page.metadata["page"]}: {len(page.page_content)} chars')

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded 447 pages
Page 0: 233 chars
Page 1: 463 chars


In [3]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # max characters per chunk
    chunk_overlap=500,    # overlap between chunks to preserve context
)

chunks = splitter.split_documents(gita_book)

print(f'Number of chunks: {len(chunks)}')
print()
for i, chunk in enumerate(chunks[:2]):
    print(f'--- Chunk {i+1} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print()

Number of chunks: 1460

--- Chunk 1 (233 chars) ---
i 
 
 
 
 
The Bhagavad Gita 
Based on HH Sri Raghavendra Teertha’s Gita Vivruti 
& 
Lectures by HH Sri Vidyasagara Madhava Teertha 
 
 
 
Compiled By 
Dr. Giridhar Boray 
 
 
 
 
 
 
 
TIRUMALA TIRUPATI DEVASTHANAMS  
TIRUPATI 
2023

--- Chunk 2 (463 chars) ---
iijkhh 
  
 
 
The Bhagavad Gita 
(Based on HH Sri Raghavendra Teertha’s Gita Vivruti) 
 
 
Compiled By 
Dr. Giridhar Boray 
 
T.T.D. Religious Publications Series No.1451 
 All Rights Reserved 
 
First Edition : 2023 
 
Copies : 500 
 
Published  by : 
Sri A.V. Dharma Reddy, IDES 
Executive Officer, 
Tirumala Tirupati Devasthanams, 
Tirupati. 
 
D.T.P: 
Publications Division, 
T.T.D, Tirupati. 
 
Printed at : 
Tirumala Tirupati Devasthanams Press, 
Tirupati.



In [4]:
!pip install langchain langchain-core langchain-ollama langchain-community chromadb --quiet


In [5]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='llama3.2')

# Create vector store from documents
# This embeds all documents and stores them in Chroma
vectorstore = Chroma(
    #documents=chunks,
    embedding_function=embeddings,
    persist_directory='./chroma_db'  # save to disk
)

print(f'Vector store created with {vectorstore._collection.count()} documents')


C:\Users\SEEDA MANASWI\AppData\Local\Temp\ipykernel_8332\215403584.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Vector store created with 5984 documents


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print('Vector store ready.')


Vector store ready.


In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model='llama3.2', temperature=0)

prompt = ChatPromptTemplate.from_template("""
"You are a scholar of Gita. Answer only from the given context."
"If the answer is not in the context, say 'This is not mentioned in the text."
Context:{context}
Question: {question}
Answer:
""")

def format_docs(docs):
    result = '\n\n'.join(doc.page_content for doc in docs)
    return result

# RAG chain using LCEL
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready.')

RAG chain ready.


In [9]:
questions =[ """
    What is the Bhagavad Gita and which larger epic is it a part of?
    Who are the Pandavas and the Kauravas?
    What event led to the Pandavas losing their kingdom to the Kauravas?
    After completing their exile, what did the Pandavas request from the Kauravas?
    What choice did Krishna offer to both sides before the war?
    """]
 #   print()
for q in questions:
    docs = vectorstore.similarity_search(q, k=5)  # use the vectorstore directly
    print(f"\nQUESTION: {q}")
    print("Retrieved Chunks:")
    for i, doc in enumerate(docs):
        print(f"Chunk {i+1}: {doc.page_content[:300]}...")  # first 300 chars
        print("-"*50)
    answer = rag_chain.invoke(q)
    print(f'A: {answer}')




QUESTION: 
    What is the Bhagavad Gita and which larger epic is it a part of?
    Who are the Pandavas and the Kauravas?
    What event led to the Pandavas losing their kingdom to the Kauravas?
    After completing their exile, what did the Pandavas request from the Kauravas?
    What choice did Krishna offer to both sides before the war?
    
Retrieved Chunks:
Chunk 1: What is this field  of activity? How is it described? What are its 
transformations? Who governs them  (referring to kshetrajna)? How is He 
described? What are His influences? Let Me answer these in brief. (13.4) 
 
Comments: The Lord now resolves to provide details of the field  of 
activity in te...
--------------------------------------------------
Chunk 2: What is this field  of activity? How is it described? What are its 
transformations? Who governs them  (referring to kshetrajna)? How is He 
described? What are His influences? Let Me answer these in brief. (13.4) 
 
Comments: The Lord now resolves to provide 

In [10]:
question = """
    1. What does Krishna say about action without attachment?
    2. What is the nature of the soul according to the Gita?
    3. What does the Gita say about a person of steady wisdom?
    4. What are the three modes of nature?
    5. What does Krishna say about those who worship other gods?
    6.Who was the 14th-century founder of the Madhava Teertha pontificate and what philosophy did he follow?
"""

# Without RAG
plain_answer = llm.invoke(question).content

# With RAG
rag_answer = rag_chain.invoke(question)

print('WITHOUT RAG:')
print(plain_answer)
print()
print('WITH RAG:')
print(rag_answer)


WITHOUT RAG:
I'll do my best to provide answers based on my knowledge cutoff of December 2023.

1. According to the Bhagavad Gita, Krishna says that action without attachment is a key principle for achieving spiritual growth and self-realization. He advises Arjuna to perform his duties (actions) without being attached to their outcomes, as this will help him maintain equanimity and detachment from the results of his actions.

2. The Bhagavad Gita describes the nature of the soul as eternal, immutable, and all-pervading. It states that the soul is a spark of the divine and is not bound by the limitations of the physical body or the material world. According to the Gita, the soul is the true self, which is beyond birth, death, and decay.

3. The Bhagavad Gita describes a person of steady wisdom as one who has achieved spiritual growth through self-realization and has developed a deep understanding of the nature of reality. Such an individual is characterized by their equanimity, detachme

    1. What chunk size worked best and why?
     I have used 1000 as chunk size and overlap 500. it is large enough to preserve meaningful sentences and context
    2. Did the retriever ever return irrelevant chunks? Give an example.
        Yes — even with semantic search, sometimes irrelevant or loosely related chunks are returned. I asked a question, and even though it said the information wasn’t available, it still gave an irrelevant answer.
    3. What would break this system? What are its limitations?

